# IndoBERT Pantun Core Pipeline (Colab)

Versi ini adalah **core pipeline** tanpa patch eksperimen.

Alur:
1. IndoBERT token labeling
2. Routing label -> jalur fonemisasi
3. CRF BI untuk pemenggalan suku kata dari fonem
4. Ekstraksi rima akhir + output ringkas pantun


In [ ]:
!pip -q install transformers sentencepiece accelerate g2p-id g2p-en nltk sklearn-crfsuite joblib pyphen

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

# UBAH sesuai lokasi Anda
MODEL_DIR = '/content/drive/MyDrive/Models/indobert-pantun-ner'
CRF_BI_MODEL_PATH = '/content/drive/MyDrive/Models/crf_layer3_bi.joblib'


In [ ]:
import os
import re
import joblib
import torch
import nltk
import pyphen
from functools import lru_cache
from transformers import AutoTokenizer, AutoModelForTokenClassification
from g2p_id import G2P as G2P_ID
from g2p_en import G2p as G2P_EN


def _ensure_nltk_resource(resource_path: str, package_name: str):
    try:
        nltk.data.find(resource_path)
    except LookupError:
        nltk.download(package_name, quiet=True)


_ensure_nltk_resource('corpora/cmudict', 'cmudict')
_ensure_nltk_resource('tokenizers/punkt', 'punkt')
_ensure_nltk_resource('tokenizers/punkt_tab', 'punkt_tab')
_ensure_nltk_resource('taggers/averaged_perceptron_tagger_eng', 'averaged_perceptron_tagger_eng')
_ensure_nltk_resource('taggers/averaged_perceptron_tagger', 'averaged_perceptron_tagger')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForTokenClassification.from_pretrained(MODEL_DIR).to(device)
model.eval()
id2label = model.config.id2label
print('Labels:', id2label)

crf_bi_model = None
if CRF_BI_MODEL_PATH and os.path.exists(CRF_BI_MODEL_PATH):
    crf_bi_model = joblib.load(CRF_BI_MODEL_PATH)
    print('[CRF3] Loaded:', CRF_BI_MODEL_PATH)
else:
    print('[CRF3] Not found:', CRF_BI_MODEL_PATH)

g2p_id_model = G2P_ID()
g2p_en_model = G2P_EN()
pyphen_en_dict = pyphen.Pyphen(lang='en_US')


In [ ]:
LABELS_ASING = {'B-ASG', 'I-ASG'}
LABELS_SINGKATAN = {'B-SKT', 'I-SKT'}
LABELS_DISEMVOWEL = {'B-DSM', 'I-DSM'}
LABELS_TYPO = {'B-TYP', 'I-TYP'}
LABELS_VALID = {'O', *LABELS_ASING, *LABELS_SINGKATAN, *LABELS_DISEMVOWEL, *LABELS_TYPO}

VOWELS = set('aiueo')
COMMON_DIGRAPHS = ('ng', 'ny', 'kh', 'sy', 'br', 'tr', 'kr', 'pr', 'gr', 'dr', 'st', 'sk')
CRF3_VOWELS = {'a', 'i', 'u', 'e', 'o', 'ə', 'ɛ', 'ɪ', 'ʊ', 'ɔ', 'æ', 'ɐ', 'ɒ', 'ɑ', 'ʌ', 'ɝ'}
CRF3_DIPHTHONGS = ('eɪ', 'aɪ', 'aʊ', 'ɔɪ', 'oʊ')

ALPHABET_EJA_MAP = {
    'a': 'a', 'b': 'be', 'c': 'ce', 'd': 'de', 'e': 'e', 'f': 'ef', 'g': 'ge',
    'h': 'ha', 'i': 'i', 'j': 'je', 'k': 'ka', 'l': 'el', 'm': 'em', 'n': 'en',
    'o': 'o', 'p': 'pe', 'q': 'ki', 'r': 'er', 's': 'es', 't': 'te', 'u': 'u',
    'v': 've', 'w': 'we', 'x': 'eks', 'y': 'ye', 'z': 'zet'
}

ARPABET_TO_IPA = {
    'AA': 'a', 'AE': 'æ', 'AH': 'ə', 'AO': 'ɔ', 'AW': 'aʊ', 'AY': 'aɪ',
    'B': 'b', 'CH': 'tʃ', 'D': 'd', 'DH': 'ð', 'EH': 'ɛ', 'ER': 'ɝ', 'EY': 'eɪ',
    'F': 'f', 'G': 'g', 'HH': 'h', 'IH': 'ɪ', 'IY': 'i', 'JH': 'dʒ', 'K': 'k',
    'L': 'l', 'M': 'm', 'N': 'n', 'NG': 'ŋ', 'OW': 'oʊ', 'OY': 'ɔɪ', 'P': 'p',
    'R': 'r', 'S': 's', 'SH': 'ʃ', 'T': 't', 'TH': 'θ', 'UH': 'ʊ', 'UW': 'u',
    'V': 'v', 'W': 'w', 'Y': 'j', 'Z': 'z', 'ZH': 'ʒ'
}


def normalize_token(token: str) -> str:
    t = token.lower().strip()
    t = ''.join(ch for ch in t if ch.isalpha())
    return re.sub(r'(.)\1{2,}', r'\1\1', t)


def route_by_label(label: str) -> str:
    norm = str(label).upper().strip()
    if norm in LABELS_SINGKATAN:
        return 'RUMUS_A_FONEMISASI_ALFABETIS'
    if norm in LABELS_DISEMVOWEL or norm in LABELS_TYPO:
        return 'RUMUS_B_C_FONEMISASI_DISEMVOWELLING'
    if norm in LABELS_ASING:
        return 'FAILSAFE_G2P_ASING'
    return 'CRF1_G2P_KATA_BAKU'


def _clean_input_words(text: str) -> list[str]:
    words = []
    for raw in str(text).strip().split():
        cleaned = re.sub(r"^[^\\w]+|[^\\w]+$", "", raw)
        if cleaned:
            words.append(cleaned)
    return words


def predict_token_labels(text: str):
    words = _clean_input_words(text)
    if not words:
        return []
    enc = tokenizer(words, is_split_into_words=True, return_tensors='pt', truncation=True, padding=False)
    enc = {k: v.to(device) for k, v in enc.items()}

    with torch.no_grad():
        logits = model(**enc).logits
        probs = torch.softmax(logits, dim=-1)[0]
        pred_ids = torch.argmax(logits, dim=-1)[0].tolist()

    word_ids = tokenizer(words, is_split_into_words=True).word_ids()
    rows = []
    cur = None
    for idx, widx in enumerate(word_ids):
        if widx is None:
            continue
        if widx != cur:
            cur = widx
            top2 = torch.topk(probs[idx], k=min(2, probs[idx].shape[-1]))
            conf = float(top2.values[0].item())
            second = float(top2.values[1].item()) if top2.values.shape[0] > 1 else 0.0
            label = id2label[pred_ids[idx]]
            rows.append({
                'token': words[widx],
                'label': label,
                'confidence': conf,
                'margin': conf - second,
                'route': route_by_label(label),
            })
    return rows


In [ ]:
def _coerce_g2p_output_to_str(raw) -> str:
    if isinstance(raw, tuple):
        return _coerce_g2p_output_to_str(raw[0])
    if isinstance(raw, list):
        if raw and isinstance(raw[0], list):
            raw = [ph for sub in raw for ph in sub if ph]
        return ' '.join(str(part) for part in raw if part)
    return str(raw).strip()


@lru_cache(maxsize=8192)
def g2p_id_phoneme(word: str) -> str:
    if not word:
        return ''
    try:
        res = g2p_id_model.to_phoneme(word)
    except AttributeError:
        try:
            res = g2p_id_model(word)
        except Exception:
            res = g2p_id_model.convert(word)
    return _coerce_g2p_output_to_str(res).replace(' ', '')


def arpabet_to_ipa(arpabet_tokens: list[str]) -> str:
    ipa_tokens = []
    for token in arpabet_tokens:
        base = re.sub(r'\d', '', token).upper()
        if not base:
            continue
        ipa_tokens.append(ARPABET_TO_IPA.get(base, base.lower()))
    return ''.join(ipa_tokens)


@lru_cache(maxsize=8192)
def g2p_foreign_phoneme(word: str) -> str:
    if not word:
        return ''
    raw = g2p_en_model(word)
    phones = [p for p in raw if p and p != ' ' and p != '<unk>']
    ipa = arpabet_to_ipa(phones)
    return ipa if ipa else g2p_id_phoneme(word)


def singkatan_to_ejaan(word: str) -> str:
    clean = ''.join(ch for ch in word.lower() if ch.isalpha())
    return ' '.join(ALPHABET_EJA_MAP.get(ch, ch) for ch in clean)


def rumus_a_singkatan_simple(word: str) -> str:
    ejaan = singkatan_to_ejaan(word)
    if not ejaan:
        return ''
    return g2p_id_phoneme(ejaan)


def _max_consonant_run(t: str) -> int:
    best = 0
    run = 0
    for ch in t:
        if ch in VOWELS:
            run = 0
        else:
            run += 1
            best = max(best, run)
    return best


def _has_common_cluster(t: str) -> bool:
    return any(dg in t for dg in COMMON_DIGRAPHS)


def _is_content_like_disemvowelling(t: str) -> bool:
    vowel_count = sum(ch in VOWELS for ch in t)
    cons_run = _max_consonant_run(t)
    if vowel_count == 0:
        return True
    if vowel_count <= 1 and len(t) >= 4:
        return True
    if cons_run >= 3:
        return True
    if cons_run >= 2 and _has_common_cluster(t):
        return True
    return False


def rumus_c_clamping(word: str, force_two_syllable: bool = False) -> str:
    t = word.lower().strip()
    if force_two_syllable:
        return 'vone' if t and t[-1] in VOWELS else 'vonem'
    if len(t) <= 2:
        return 'von'
    return 'vone' if t and t[-1] in VOWELS else 'vonem'


def rumus_b_phonotactic_mapping(word: str) -> str:
    t = ''.join(ch for ch in word.lower() if ch.isalpha())
    if not t:
        return 'von'
    vowel_count = sum(ch in VOWELS for ch in t)
    cons_run = _max_consonant_run(t)
    has_cluster = _has_common_cluster(t)
    if vowel_count == 0:
        sonorants = set('rlmnwy')
        if len(t) == 3 and any(ch in sonorants for ch in t):
            return 'vonemi'
        if len(t) <= 4:
            syll_est = 2
        elif has_cluster or cons_run >= 4:
            syll_est = 3
        else:
            syll_est = 2
        return {2: 'vonem', 3: 'vonemis', 4: 'vonemisa'}[syll_est]

    compact = t
    for dg in COMMON_DIGRAPHS:
        compact = compact.replace(dg, 'X')
    base_units = len(compact)
    has_coda = t[-1] in 'bcdfghjklmnpqrstvwxyz'
    if has_coda and base_units > 1:
        base_units -= 1
    syll_est = max(2, min(4, (base_units + 1) // 2))
    return {2: 'vonem', 3: 'vonemis', 4: 'vonemisa'}[syll_est]


def rumus_bc_for_dsm_typo(token: str, label: str) -> dict:
    norm_label = label.upper().strip()
    normalized_token = normalize_token(token)
    if normalized_token in {'gw', 'gue', 'gua'}:
        spoken = 'gue' if normalized_token == 'gw' else normalized_token
        return {'normalized_token': spoken, 'rule_used': 'C', 'phoneme': rumus_c_clamping(spoken, force_two_syllable=True)}
    if norm_label in LABELS_TYPO and len(normalized_token) <= 3:
        return {'normalized_token': normalized_token, 'rule_used': 'C', 'phoneme': rumus_c_clamping(normalized_token, force_two_syllable=True)}
    if len(normalized_token) <= 2 and all(ch not in VOWELS for ch in normalized_token):
        return {'normalized_token': normalized_token, 'rule_used': 'C', 'phoneme': rumus_c_clamping(normalized_token)}
    if _is_content_like_disemvowelling(normalized_token):
        return {'normalized_token': normalized_token, 'rule_used': 'B', 'phoneme': rumus_b_phonotactic_mapping(normalized_token)}
    return {'normalized_token': normalized_token, 'rule_used': 'C', 'phoneme': rumus_c_clamping(normalized_token)}


def acoustic_switchboard(token: str, label: str) -> dict:
    norm = label.upper().strip()
    cleaned = normalize_token(token)
    if norm in LABELS_SINGKATAN:
        ejaan = singkatan_to_ejaan(token)
        return {'switch_route': 'A_SINGKATAN_G2P_ID', 'switch_input': ejaan, 'phoneme': rumus_a_singkatan_simple(token)}
    if norm in LABELS_ASING:
        return {'switch_route': 'ASG_G2P_EN_IPA', 'switch_input': cleaned, 'phoneme': g2p_foreign_phoneme(cleaned)}
    if norm in LABELS_DISEMVOWEL or norm in LABELS_TYPO:
        bc = rumus_bc_for_dsm_typo(token, norm)
        return {'switch_route': f"BC_{bc['rule_used']}", 'switch_input': bc['normalized_token'], 'phoneme': bc['phoneme'], 'bc_info': bc}
    return {'switch_route': 'O_G2P_ID', 'switch_input': cleaned, 'phoneme': g2p_id_phoneme(cleaned)}


In [ ]:
def phoneme_to_units(phoneme: str) -> list[str]:
    text = str(phoneme).strip()
    if not text:
        return []
    if ' ' in text:
        return [u for u in text.split() if u]

    chars = [ch for ch in text if not ch.isspace()]
    units = []
    i = 0
    while i < len(chars):
        if i + 1 < len(chars):
            pair = chars[i] + chars[i + 1]
            if pair in CRF3_DIPHTHONGS:
                units.append(pair)
                i += 2
                continue
        units.append(chars[i])
        i += 1
    return units


def _crf3_is_vowel(phone: str) -> bool:
    return phone in CRF3_VOWELS


def _crf3_phone_features(seq: list[str], i: int) -> dict:
    cur = seq[i]
    prev = seq[i - 1] if i > 0 else '<BOS>'
    nxt = seq[i + 1] if i < len(seq) - 1 else '<EOS>'
    return {
        'bias': 1.0,
        'phone': cur,
        'prev_phone': prev,
        'next_phone': nxt,
        'bigram_prev_cur': f'{prev}+{cur}',
        'bigram_cur_next': f'{cur}+{nxt}',
        'is_vowel': _crf3_is_vowel(cur),
        'prev_is_vowel': False if prev == '<BOS>' else _crf3_is_vowel(prev),
        'next_is_vowel': False if nxt == '<EOS>' else _crf3_is_vowel(nxt),
        'pos': i,
        'rev_pos': len(seq) - 1 - i,
        'len_seq': len(seq),
        'bos': i == 0,
        'eos': i == len(seq) - 1,
    }


def predict_bi_from_phoneme(phoneme: str, model=None):
    mdl = model if model is not None else crf_bi_model
    units = phoneme_to_units(phoneme)
    if not units:
        return {'units': [], 'bi': '', 'valid': False}
    if mdl is None:
        return {'units': units, 'bi': 'B' + 'I' * (len(units) - 1), 'valid': False}
    feats = [_crf3_phone_features(units, i) for i in range(len(units))]
    if hasattr(mdl, 'predict_single'):
        pred = mdl.predict_single(feats)
    else:
        pred = mdl.predict([feats])[0]
    bi = ''.join(pred)
    return {'units': units, 'bi': bi, 'valid': len(bi) == len(units) and bi.count('B') > 0}


def decode_bi_to_syllables(units: list[str], bi_pattern: str) -> list[list[str]]:
    if not units:
        return []
    if not bi_pattern or len(bi_pattern) != len(units):
        return [units]
    out = []
    cur = []
    for u, tag in zip(units, bi_pattern):
        if tag == 'B' and cur:
            out.append(cur)
            cur = [u]
        else:
            cur.append(u)
    if cur:
        out.append(cur)
    return out


def extract_rhyme_tail(syllable_text: str) -> str:
    if not syllable_text:
        return ''
    m = re.search(r'([aiueoəɛɪæɔʊɝ][^aiueoəɛɪæɔʊɝ]*)$', syllable_text)
    return m.group(1) if m else syllable_text[-2:]


def split_asg_with_pyphen(token: str) -> list[str]:
    dic = globals().get('pyphen_en_dict')
    clean = normalize_token(token)
    if dic is None or not clean:
        return []
    inserted = dic.inserted(clean)
    parts = [p for p in inserted.split('-') if p]
    return parts if parts else [clean]


def analyze_line_prosody_full(text: str) -> dict:
    rows = predict_with_routing(text)
    token_reports = []
    for row in rows:
        ac = acoustic_switchboard(row['token'], row['final_label'])
        ph = ac.get('phoneme', '')
        bi = predict_bi_from_phoneme(ph)
        syll_units = decode_bi_to_syllables(bi['units'], bi['bi'])
        syllables = [''.join(s) for s in syll_units]
        syllable_source = 'CRF_BI'
        if row['final_label'] in LABELS_SINGKATAN:
            letter_syllables = [u for u in singkatan_to_ejaan(row['token']).split() if u]
            if letter_syllables:
                syllables = letter_syllables
                syllable_source = 'SKT_PER_LETTER'
        elif row['final_label'] in LABELS_ASING:
            asg_syllables = split_asg_with_pyphen(ac.get('switch_input') or row['token'])
            if asg_syllables:
                syllables = asg_syllables
                syllable_source = 'ASG_PYPHEN_EN'
        rhyme_tail = extract_rhyme_tail(syllables[-1] if syllables else '')
        token_reports.append({
            'token': row['token'],
            'raw_label': row['label'],
            'final_label': row['final_label'],
            'route': row['route'],
            'switch': ac.get('switch_route'),
            'confidence': row['confidence'],
            'margin': row['margin'],
            'phoneme': ph,
            'units': bi['units'],
            'bi': bi['bi'],
            'bi_valid': bi['valid'],
            'syllables': syllables,
            'syllable_source': syllable_source,
            'syllable_count': len(syllables),
            'token_rhyme_tail': rhyme_tail,
            'bc_info': ac.get('bc_info'),
        })
    line_syllables = sum(t['syllable_count'] for t in token_reports)
    line_rhyme = token_reports[-1]['token_rhyme_tail'] if token_reports else ''
    return {'text': text, 'tokens': token_reports, 'line_syllable_count': line_syllables, 'line_rhyme_tail': line_rhyme}


def predict_with_routing(text: str):
    rows = predict_token_labels(text)
    out = []
    for r in rows:
        final_label = r['label'] if r['label'] in LABELS_VALID else 'O'
        out.append({
            'token': r['token'],
            'label': r['label'],
            'final_label': final_label,
            'route': route_by_label(final_label),
            'confidence': r['confidence'],
            'margin': r['margin'],
        })
    return out


def infer_rhyme_scheme(line_reports: list[dict]) -> str:
    keys = [rep.get('line_rhyme_tail', '') for rep in line_reports]
    mapping = {}
    seq = []
    next_char = ord('a')
    for k in keys:
        if k not in mapping:
            mapping[k] = chr(next_char)
            next_char += 1
        seq.append(mapping[k])
    return '-'.join(seq)


def analyze_pantun_multiline(text: str) -> dict:
    lines = [ln.strip() for ln in text.replace('\\n', '\n').splitlines() if ln.strip()]
    line_reports = [analyze_line_prosody_full(ln) for ln in lines]
    return {
        'raw_text': text,
        'line_count': len(line_reports),
        'lines': line_reports,
        'total_syllables_all_lines': sum(rep['line_syllable_count'] for rep in line_reports),
        'rhyme_scheme': infer_rhyme_scheme(line_reports),
    }


def build_pantun_compact_output(report: dict) -> dict:
    lines = report.get('lines', [])
    compact = {
        'skema_rima': report.get('rhyme_scheme', ''),
        'rima_akhir_baris_1': '', 'rima_akhir_baris_2': '', 'rima_akhir_baris_3': '', 'rima_akhir_baris_4': '',
        'jumlah_suku_baris_1': 0, 'jumlah_suku_baris_2': 0, 'jumlah_suku_baris_3': 0, 'jumlah_suku_baris_4': 0,
    }
    for i, rep in enumerate(lines[:4], start=1):
        compact[f'rima_akhir_baris_{i}'] = rep.get('line_rhyme_tail', '')
        compact[f'jumlah_suku_baris_{i}'] = rep.get('line_syllable_count', 0)
    return compact


def print_line_prosody_analysis(line_report: dict) -> None:
    print(f"line: {line_report.get('text', '')}")
    print(f"line_rhyme_tail={line_report.get('line_rhyme_tail', '')} | line_syllable_count={line_report.get('line_syllable_count', 0)}")
    print('--- token detail ---')
    for t in line_report.get('tokens', []):
        print(
            f"token={t.get('token','')} | raw={t.get('raw_label','')} | final={t.get('final_label','')} "
            f"| route={t.get('route','')} | switch={t.get('switch','')} | conf={t.get('confidence',0):.3f} margin={t.get('margin',0):.3f}"
        )
        print(f"     phoneme={t.get('phoneme','')}")
        print(f"     units={t.get('units',[])} (n={len(t.get('units',[]))})")
        print(f"     bi={t.get('bi','')} | bi_valid={t.get('bi_valid', False)}")
        print(f"     syllables={t.get('syllables',[])} | syllable_count={t.get('syllable_count',0)} | source={t.get('syllable_source','')}")
        if t.get('bc_info'):
            print(f"     bc_info={t.get('bc_info')}")
        print(f"     token_rhyme_tail={t.get('token_rhyme_tail','')}")
    print('')


def print_pantun_multiline_analysis(report: dict, verbose: bool = False) -> None:
    print('=== PANTUN ANALYSIS ===')
    print(f"line_count={report.get('line_count',0)} | total_syllables={report.get('total_syllables_all_lines',0)} | rhyme_scheme={report.get('rhyme_scheme','')}")
    for i, line in enumerate(report.get('lines', []), start=1):
        print(f"baris_{i}: rima={line.get('line_rhyme_tail','')} suku={line.get('line_syllable_count',0)}")
        if verbose:
            print_line_prosody_analysis(line)
    print('')
    print('compact output:')
    print(build_pantun_compact_output(report))


In [ ]:
# Contoh pakai
text = "lagi ngeliat ilc sekarang\npetinggi sunda empire bikin ngakak\ncintaku jangan kau buang\naku gamau jantungku berhenti mendadak"

report = analyze_pantun_multiline(text)
print_pantun_multiline_analysis(report, verbose=True)
